# MaxViT — From Scratch in PyTorch
**Paper:** MaxViT: Multi-Axis Vision Transformer (Tu et al., ECCV 2022)

| Config | Value |
|--------|-------|
| Variant | Tiny |
| Stages | 4 (depths: 2,2,5,2) |
| Channels | 64,128,256,512 |
| Window Size | 8×8 |
| Input Size | 224×224 |
| Parameters | ~31M |

In [ ]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — MaxViT Architecture (raw, no torchvision)
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super(SqueezeExcitation, self).__init__()
        hidden = max(1, channels // reduction)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.se(x)


class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=4, se_reduction=4):
        super(MBConv, self).__init__()
        hidden = in_channels * expansion
        self.pre_norm = nn.BatchNorm2d(in_channels)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.GELU(),
            SqueezeExcitation(hidden, se_reduction),
            nn.Conv2d(hidden, out_channels, 1, bias=False),
        )
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1, bias=False)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return self.conv(self.pre_norm(x)) + self.shortcut(x)


class RelativePositionBias(nn.Module):
    def __init__(self, window_size, num_heads):
        super(RelativePositionBias, self).__init__()
        self.window_size = window_size
        self.num_heads   = num_heads
        seq = 2 * window_size - 1
        self.bias_table  = nn.Parameter(torch.zeros(seq * seq, num_heads))
        coords = torch.stack(torch.meshgrid(
            torch.arange(window_size), torch.arange(window_size), indexing='ij'
        ))
        coords_flat = coords.flatten(1)
        relative     = coords_flat[:, :, None] - coords_flat[:, None, :]
        relative[0] += window_size - 1
        relative[1] += window_size - 1
        relative[0] *= 2 * window_size - 1
        idx = relative.sum(0)
        self.register_buffer('relative_index', idx)

    def forward(self):
        bias = self.bias_table[self.relative_index.view(-1)]
        bias = bias.view(self.window_size**2, self.window_size**2, self.num_heads)
        return bias.permute(2, 0, 1).unsqueeze(0)


class WindowAttention(nn.Module):
    """Local window self-attention."""
    def __init__(self, dim, num_heads, window_size):
        super(WindowAttention, self).__init__()
        self.num_heads  = num_heads
        self.head_dim   = dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.qkv        = nn.Linear(dim, dim * 3)
        self.proj       = nn.Linear(dim, dim)
        self.rel_bias   = RelativePositionBias(window_size, num_heads)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale + self.rel_bias()
        attn = attn.softmax(dim=-1)
        x    = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class GridAttention(nn.Module):
    """Dilated global grid self-attention."""
    def __init__(self, dim, num_heads):
        super(GridAttention, self).__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3)
        self.proj      = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x    = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class MaxViTBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=8, mlp_ratio=4):
        super(MaxViTBlock, self).__init__()
        self.window_size = window_size
        hidden           = int(dim * mlp_ratio)

        self.mbconv = MBConv(dim, dim)

        self.norm1  = nn.LayerNorm(dim)
        self.win_attn = WindowAttention(dim, num_heads, window_size)
        self.norm2  = nn.LayerNorm(dim)
        self.mlp1   = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
        )

        self.norm3  = nn.LayerNorm(dim)
        self.grid_attn = GridAttention(dim, num_heads)
        self.norm4  = nn.LayerNorm(dim)
        self.mlp2   = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
        )

    def _partition_windows(self, x):
        B, C, H, W = x.shape
        ws = self.window_size
        x  = x.reshape(B, C, H // ws, ws, W // ws, ws)
        x  = x.permute(0, 2, 4, 3, 5, 1).reshape(-1, ws * ws, C)
        return x, B, H, W

    def _unpartition_windows(self, x, B, H, W):
        ws = self.window_size
        C  = x.shape[-1]
        x  = x.reshape(B, H // ws, W // ws, ws, ws, C)
        x  = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, H, W)
        return x

    def _partition_grid(self, x):
        B, C, H, W = x.shape
        ws = self.window_size
        x  = x.reshape(B, C, ws, H // ws, ws, W // ws)
        x  = x.permute(0, 3, 5, 2, 4, 1).reshape(-1, ws * ws, C)
        return x, B, H, W

    def _unpartition_grid(self, x, B, H, W):
        ws = self.window_size
        C  = x.shape[-1]
        x  = x.reshape(B, H // ws, W // ws, ws, ws, C)
        x  = x.permute(0, 5, 3, 1, 4, 2).reshape(B, C, H, W)
        return x

    def forward(self, x):
        x = self.mbconv(x)

        # Block (local window) attention
        tokens, B, H, W = self._partition_windows(x)
        tokens = tokens + self.win_attn(self.norm1(tokens))
        tokens = tokens + self.mlp1(self.norm2(tokens))
        x      = self._unpartition_windows(tokens, B, H, W)

        # Grid (global dilated) attention
        tokens, B, H, W = self._partition_grid(x)
        tokens = tokens + self.grid_attn(self.norm3(tokens))
        tokens = tokens + self.mlp2(self.norm4(tokens))
        x      = self._unpartition_grid(tokens, B, H, W)

        return x


class MaxViTStage(nn.Module):
    def __init__(self, in_channels, out_channels, depth, num_heads, window_size=8):
        super(MaxViTStage, self).__init__()
        self.downsample = MBConv(in_channels, out_channels)
        self.blocks     = nn.Sequential(*[
            MaxViTBlock(out_channels, num_heads, window_size)
            for _ in range(depth)
        ])

    def forward(self, x):
        return self.blocks(self.downsample(x))


class MaxViT(nn.Module):
    def __init__(self, in_channels=3, num_classes=1000,
                 depths=(2, 2, 5, 2), channels=(64, 128, 256, 512),
                 num_heads=(2, 4, 8, 16), window_size=8):
        super(MaxViT, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.GELU(),
            nn.Conv2d(channels[0], channels[0], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.GELU(),
        )

        stage_in = [channels[0]] + list(channels[:-1])
        self.stages = nn.Sequential(*[
            MaxViTStage(stage_in[i], channels[i], depths[i], num_heads[i], window_size)
            for i in range(len(depths))
        ])

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.LayerNorm(channels[-1]),
            nn.Linear(channels[-1], channels[-1]),
            nn.Tanh(),
            nn.Linear(channels[-1], num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        return self.head(x)


def maxvit_tiny(num_classes=1000):
    return MaxViT(
        num_classes = num_classes,
        depths      = (2, 2, 5, 2),
        channels    = (64, 128, 256, 512),
        num_heads   = (2, 4, 8, 16),
        window_size = 8,
    )

In [ ]:
# Cell 3 — Instantiate Model
NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = maxvit_tiny(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

In [ ]:
# Cell 4 — Forward Pass Test (batch=2, 224×224)
dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
output      = model(dummy_input)
print(f'Input  shape : {dummy_input.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
# Cell 5 — Count Parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable    = total_params - trainable_params

print(f"{'='*40}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable params  : {non_trainable:,}")
print(f"{'='*40}")

In [ ]:
# Cell 6 — Layer-wise Parameter Breakdown
print(f"{'Layer':<40} {'Shape':<30} {'Params':>10}")
print('-' * 82)
for name, param in model.named_parameters():
    print(f"{name:<40} {str(list(param.shape)):<30} {param.numel():>10,}")
print('-' * 82)
print(f"{'TOTAL':<71} {total_params:>10,}")

---
## Training

In [ ]:
# Cell 7 — Dataset & DataLoader
DATA_DIR   = './data'
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

CLASS_NAMES = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
# Cell 8 — Training Loop
EPOCHS = 20
LR     = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'maxvit_best.pth')
print('\nModel saved: maxvit_best.pth')

---
## Training Curves

In [ ]:
# Cell 9 — Training Curves (Loss & Accuracy)
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MaxViT — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

---
## Inference

In [ ]:
# Cell 10 — Inference on a Single Image
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)

    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input Image')
    labels = [CLASS_NAMES[i] for i in top_idx]
    colors = ['#F44336' if i == 0 else '#2196F3' for i in range(top_k)]
    bars   = axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    for bar, prob in zip(bars, top_probs[::-1]):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{prob*100:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig('inference_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nPredicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
# Cell 11 — Collect Validation Predictions
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')
print(f'Labels shape      : {all_labels.shape}')

In [ ]:
# Cell 12 — ROC AUC Curve (One-vs-Rest, per class)
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'MaxViT — ROC AUC Curve (Macro AUC = {macro_auc:.3f})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMacro AUC : {macro_auc:.4f}')
print('\nPer-class AUC:')
for cls, score in roc_auc_scores.items():
    print(f'  {cls:<20} {score:.4f}')